In [ ]:
import pandas as pd

df = pd.read_csv("quadrant_predictions.csv")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-colorblind')

# compute relative error if not given
df["rel_error"] = np.abs(df["pred_area_cm2"] - df["gt_area_cm2"]) / df["gt_area_cm2"]

# flag within 10%
threshold = 0.10
df["within10pct"] = df["rel_error"] <= threshold

# summary
n = len(df)
n_within = df["within10pct"].sum()

# scatter plot
fig1 = plt.figure(figsize=(10,8))
plt.scatter(df["gt_area_cm2"], df["pred_area_cm2"],
            c=df["within10pct"].map({True:"green", False:"red"}),
            alpha=0.7,
            s=100)
minv = min(df["gt_area_cm2"].min(), df["pred_area_cm2"].min())
maxv = max(df["gt_area_cm2"].max(), df["pred_area_cm2"].max())
plt.plot([minv, maxv], [minv, maxv], 'k--', label="Perfect prediction")
plt.plot([minv, maxv], [minv*(1+threshold), maxv*(1+threshold)], 'b--', label=f"+{threshold*100:.0f}% band")
plt.plot([minv, maxv], [minv*(1-threshold), maxv*(1-threshold)], 'b--', label=f"-{threshold*100:.0f}% band")
plt.xlabel("Ground truth area", fontsize=20)
plt.ylabel("Predicted area", fontsize=20)
plt.title("Predicted vs Ground Truth", fontsize=26)
plt.legend(fontsize=20)
plt.grid(True)
plt.xlim(0.00025, 0.0011)
plt.ylim(0.00025, 0.0011)
plt.tight_layout()
plt.tick_params(axis='both', which='major', labelsize=16)

# Save to file
fig1.savefig("scatter_pred_vs_gt.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig1)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.style.use('seaborn-v0_8-colorblind')

# keep values as fractions in [0,1], just *display* as percentages
rel = df["rel_error"].astype(float).dropna().to_numpy()
threshold = 0.10  # 10%

fig2, ax = plt.subplots(figsize=(5, 8), dpi=300)

ax.boxplot(
    rel,
    vert=True,
    showmeans=True,
    meanline=True,
    patch_artist=True,
    boxprops=dict(alpha=0.9),
    whiskerprops=dict(alpha=0.9),
    capprops=dict(alpha=0.9),
    medianprops=dict(linewidth=2),
    meanprops=dict(color='black', linewidth=1.5),
)

# Cosmetic: single x tick label
ax.set_xticks([1])
ax.set_xticklabels(["Relative error"], fontsize=18)

# Y axis range (robust to outliers)
y_max = float(np.percentile(rel, 99.5)) * 1.1 if rel.size else 1.0
y_max = max(y_max, rel.max() * 1.05 if rel.size else 1.0)
ax.set_ylim(0, min(1.0, y_max))  # clamp to 100%

# Threshold reference line at 10%
ax.axhline(threshold, linestyle="--", color="#1f77b4", alpha=0.8, label="10% threshold")

# Format y-axis as percentages
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))

ax.set_ylabel("Relative error (%)", fontsize=18)
ax.set_title("Relative Error Distribution", fontsize=18, pad=8)
ax.grid(axis="y", linestyle=":", alpha=0.5)
# ax.legend(loc="upper right", frameon=True, framealpha=0.95, fontsize=14)

ax.tick_params(axis='y', labelsize=14)
plt.tight_layout()
plt.show()
plt.close(fig2)


In [ ]:
(df["rel_error"].mean() * 100).round(2), (df["rel_error"].median() * 100).round(2)

In [ ]:
import math
from pathlib import Path
from typing import List, Tuple, Optional, Iterable, Union, Dict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.patches import Polygon

# -------------------------
# Prediction helpers (your logic)
# -------------------------
def to_mask2d(m_or_path: Union[str, np.ndarray]) -> np.ndarray:
    m = np.asarray(m_or_path)
    m = (np.squeeze(m) > 0).astype(np.uint8)
    if m.ndim != 2:
        raise ValueError(f"Expected 2D mask, got shape {m.shape}")
    return m

def point_bottom_right(m: np.ndarray) -> Tuple[int, int]:
    ys, xs = np.where(m == 1)
    if ys.size == 0: raise ValueError("Mask is empty.")
    i = (ys.astype(np.int64) * xs.astype(np.int64)).argmax()
    return int(ys[i]), int(xs[i])

def point_top_left(m: np.ndarray) -> Tuple[int, int]:
    ys, xs = np.where(m == 1)
    keep = (ys > 0) & (xs > 0)
    ys, xs = ys[keep], xs[keep]
    if ys.size == 0: raise ValueError("No white pixels with x>0 and y>0.")
    sums = ys.astype(np.int64) + xs.astype(np.int64)
    min_sum = sums.min()
    idx = np.where(sums == min_sum)[0]
    y_sel = int(ys[idx].min())
    x_sel = int(xs[idx][ys[idx] == y_sel].min())
    return y_sel, x_sel

def point_top_right(m: np.ndarray) -> Tuple[int, int]:
    ys, xs = np.where(m == 1)
    diffs = xs.astype(np.int64) - ys.astype(np.int64)
    best = diffs.max()
    keep = (diffs == best)
    xs_t, ys_t = xs[keep], ys[keep]
    x_sel = int(xs_t.max())
    y_sel = int(ys_t[xs_t == x_sel].min())
    return y_sel, x_sel

def point_bottom_left(m: np.ndarray) -> Tuple[int, int]:
    ys, xs = np.where(m == 1)
    diffs = ys.astype(np.int64) - xs.astype(np.int64)
    best = diffs.max()
    keep = (diffs == best)
    ys_t, xs_t = ys[keep], xs[keep]
    y_sel = int(ys_t.max())
    x_sel = int(xs_t[ys_t == y_sel].min())
    return y_sel, x_sel

def find_four_points(mask_2d: Union[str, np.ndarray]) -> Dict[str, Tuple[int,int]]:
    m = to_mask2d(mask_2d)
    return {
        "TL": point_top_left(m),
        "TR": point_top_right(m),
        "BR": point_bottom_right(m),
        "BL": point_bottom_left(m),
    }

def overlay_color_mask(img_rgb: np.ndarray, mask_bool: np.ndarray,
                       color=(0, 0, 255), alpha: float = 0.35) -> np.ndarray:
    out = img_rgb.astype(np.float32)
    c = np.array(color, dtype=np.float32)[None, None, :]
    m = mask_bool.astype(np.float32)[..., None]
    out = out * (1.0 - alpha * m) + c * (alpha * m)
    return np.clip(out, 0, 255).astype(np.uint8)

# -------------------------
# Ground-truth (CPC) helpers (your logic)
# -------------------------
import re
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}
FLOAT = r"[+-]?(?:\d+(?:\.\d+)?|\.\d+)"
PAIR_RE = re.compile(rf'^\s*"?\s*({FLOAT})\s*"?\s*,\s*"?\s*({FLOAT})\s*"?\s*$')
NUMS_RE = re.compile(rf"{FLOAT}")

def _read_text(path: Path) -> str:
    for enc in ("utf-8-sig", "latin-1"):
        try: return path.read_text(encoding=enc, errors="strict")
        except Exception: pass
    return path.read_text(encoding="utf-8", errors="ignore")

def parse_cpc(path: Path):
    text = _read_text(path)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip() != ""]
    if not lines:
        raise ValueError(f"Empty CPC: {path}")
    nums = [float(s) for s in NUMS_RE.findall(lines[0])]
    if len(nums) < 4:
        raise ValueError(f"Unexpected CPC header format: {path}")
    work_w, work_h = float(nums[-4]), float(nums[-3])
    roi = []
    i = 1
    while i < len(lines) and len(roi) < 4:
        m = PAIR_RE.match(lines[i])
        if m:
            roi.append((float(m.group(1)), float(m.group(2))))
        i += 1
    if len(roi) != 4:
        raise ValueError(f"Could not find 4 ROI corners in {path}")
    return work_w, work_h, roi

def _scale_points_to_pixels(roi, work_w, work_h, img_w, img_h):
    sx, sy = img_w / work_w, img_h / work_h
    return [(x * sx, y * sy) for (x, y) in roi]

# -------------------------
# Main function
# -------------------------
from automated_underwater_area_estimation.segmentation_quadrant.model import QuadrantSegmentationModel

def juxtapose_prediction_vs_gt(
    image_paths: List[Path | str],
    cpcs_dir: Path | str,
    *,
    model: Optional[QuadrantSegmentationModel] = None,
    alpha: float = 0.35,
    overlay_color=(0, 0, 255),               # prediction overlay color (RGB)
    pt_color="#FFBF00", side_color="#FFBF00", diag_color="#FFBF00",
    figsize_per_row: Tuple[float, float] = (12.0, 4.0),  # width × height per row (2 columns total)
    titlefont: int = 16,
    show_titles: bool = True,
):
    """
    For each image in `image_paths`, draws two panels in a single figure row:
      Left : Prediction overlay with quadrant corner points + lines
      Right: Ground-truth overlay from matching .cpc file

    Assumes a GT CPC file named <stem>.cpc lives in `cpcs_dir`.

    Returns:
        (fig, axes): Matplotlib figure and axes array of shape (N, 2)
    """
    paths = [Path(p) for p in image_paths]
    cpcs_dir = Path(cpcs_dir)
    if model is None:
        model = QuadrantSegmentationModel()

    n = len(paths)
    if n == 0:
        raise ValueError("No image paths provided.")
    fig, axes = plt.subplots(
        nrows=n, ncols=2, figsize=(figsize_per_row[0], figsize_per_row[1] * n),
        dpi=150, squeeze=False
    )

    for row, img_path in enumerate(paths):
        stem = img_path.stem
        cpc_path = cpcs_dir / f"{stem}.cpc"
        if not img_path.exists():
            raise FileNotFoundError(f"Image not found: {img_path}")
        if not cpc_path.exists():
            raise FileNotFoundError(f"CPC not found: {cpc_path}")

        # --- load image ---
        img = np.asarray(Image.open(img_path).convert("RGB"))

        # --- PREDICTION PANEL (left) ---
        axL = axes[row, 0]
        # predict mask
        pred = model.segment_image(Image.fromarray(img))
        if not isinstance(pred, np.ndarray):
            try: pred = pred.detach().cpu().numpy()
            except Exception: pred = np.asarray(pred)
        if pred.ndim == 3:  # (C,H,W) or (H,W,1)
            pred = pred.max(axis=0) if pred.shape[0] in (1, 3) else pred[..., 0]
        mask_bool = (pred > 0.5)

        overlay_pred = overlay_color_mask(img, mask_bool, color=overlay_color, alpha=alpha)
        axL.imshow(overlay_pred)

        # find 4 extreme points & draw lines
        m2d = to_mask2d(mask_bool.astype(np.uint8))
        pts = find_four_points(m2d)
        # points
        for k, (y, x) in pts.items():
            axL.scatter([x], [y], s=150, facecolor="white", edgecolor="black", zorder=4)
            axL.scatter([x], [y], s=70, color=pt_color, zorder=5)
        # sides and diagonals
        sides = [("TR", "TL"), ("TR", "BR"), ("BL", "BR"), ("TL", "BL")]
        diags = [("TL", "BR"), ("TR", "BL")]
        def _seg(a, b, color, lw=2.2, z=2):
            (y1, x1), (y2, x2) = pts[a], pts[b]
            axL.plot([x1, x2], [y1, y2], color=color, lw=lw, alpha=0.9, zorder=z)
        for a, b in sides: _seg(a, b, side_color, 2.2, 2)
        for a, b in diags: _seg(a, b, diag_color, 2.4, 1)

        if show_titles:
            axL.set_title("Prediction", fontsize=titlefont)
        axL.axis("off")

        # --- GROUND-TRUTH PANEL (right) ---
        axR = axes[row, 1]
        work_w, work_h, roi = parse_cpc(cpc_path)
        h, w = img.shape[0], img.shape[1]
        roi_px = _scale_points_to_pixels(roi, work_w, work_h, w, h)

        axR.imshow(img)
        poly = Polygon(np.array(roi_px, dtype=np.float32), closed=True, fill=False,
                       linewidth=4, color="#d95f02")
        axR.add_patch(poly)
        xs, ys = zip(*roi_px)
        axR.scatter(xs, ys, s=70, color="#1b9e77")
        if show_titles:
            axR.set_title("Ground truth", fontsize=titlefont)
        axR.axis("off")

    plt.tight_layout(pad=0.2)
    return fig, axes


In [ ]:
# List of images you want to compare (any order)
images = df.query("rel_error > 0.1")["image_path"].tolist()
images = [image.replace("automated_underwater_area_estimation", "..") for image in images]
# Folder that contains the matching .cpc files (same stem names)
cpcs_dir = "../data_preprocessed/IBF/cpcs"

fig, axes = juxtapose_prediction_vs_gt(
    images, cpcs_dir,
    alpha=0.35,
    overlay_color=(0,0,255),   # blue overlay for prediction
    titlefont=16,
    show_titles=False
)
plt.show()
